# MDLM 实验 Notebook — 扩散过程可视化 & 采样器对比

**Spring 2026《随机过程》课程项目**

基于 MDLM (Masked Diffusion Language Model, Sahoo et al., NeurIPS 2024) 的推理实验。

---

## 包含实验

| # | 实验 | 内容 |
|:-:|------|------|
| 1a | **多 Seed 可视化** | 3 个随机种子下的去噪过程对比 (seed=42, 123, 999) |
| 1b | **步数对比** | 128 / 500 / 1000 步对生成质量的影响 |
| 2 | **采样器效率对比** | ddpm_cache / ddpm / analytic 三者的速度与 PPL |

---

## 提升生成质量的技巧

MDLM 的生成质量受以下因素影响：
1. **采样步数**：越多越好，推荐 1000 步
2. **随机种子**：不同 seed 差异很大，多试几个
3. **序列长度**：256 是平衡质量和速度的选择

> ⏱ T4 GPU 上 1000 步约需 3-5 分钟

---
## 0. 安装依赖 & 设置环境

In [20]:
!python --version

Python 3.12.13


In [21]:
!nvcc -V

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [1]:
import os, sys, time, json, itertools, math, zipfile, io

# 安装依赖
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.38.2 datasets einops==0.7.0 -q
!pip install lightning==2.2.1 hydra-core==1.3.2 omegaconf==2.3.0 -q
!pip install fsspec h5py packaging==23.2 rich==13.7.1 -q
!pip install numpy==1.26.4 -q
!pip install triton -q

In [23]:
!python -c "import torch; print(torch.__version__);"

2.2.1+cu121


In [24]:
!python -c "import torch; print(torch._C._GLIBCXX_USE_CXX11_ABI)"

False


In [2]:
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu12torch2.2cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.3/191.3 MB 6.1 MB/s eta 0:00:00


In [3]:
# ============================================================
# 上传代码包
#
# 运行后会出现「选择文件」按钮，选本地 mdlm_chinese.zip
# zip 只有 ~114KB，上传很快
# ============================================================

EXTRACT_DIR = '/content/mdlm_chinese'

if os.path.exists(EXTRACT_DIR):
    print(f'代码已就绪: {EXTRACT_DIR}')
else:
    from google.colab import files
    print('请在弹出的对话框中选择 mdlm_chinese.zip ...')
    uploaded = files.upload()  # 返回 {文件名: 字节内容}
    for fname, content in uploaded.items():
        with zipfile.ZipFile(io.BytesIO(content)) as zf:
            zf.extractall(EXTRACT_DIR)
        print(f'已解压 {fname} ({len(content)/1024:.0f} KB)')

os.chdir(EXTRACT_DIR)
print(f'工作目录: {os.getcwd()}')

代码已就绪: /content/mdlm_chinese
工作目录: /content/mdlm_chinese


In [4]:
# 检查 NumPy 版本，必要时重启
import numpy as np
if np.__version__ != '1.26.4':
    print(f'NumPy 版本不匹配 ({np.__version__})，请重启运行时后重新运行')
    os.kill(os.getpid(), 9)
print(f'NumPy: {np.__version__} ✓')

# 导入 MDLM 模块
import torch
import hydra
import lightning as L
from omegaconf import DictConfig

import dataloader
import diffusion
import main
import utils

print('所有模块导入成功 ✓')

NumPy: 1.26.4 ✓
所有模块导入成功 ✓


---
## 辅助函数

In [5]:
def make_config(overrides):
    cfg_path = os.path.join(os.getcwd(), 'configs')
    print(f'配置路径: {cfg_path}')
    with hydra.initialize_config_dir(config_dir=cfg_path, version_base=None):
        return hydra.compose(config_name='config', overrides=overrides)
def run_experiment(config, tag=''):
    """运行一次生成实验，返回耗时和文本。"""
    L.seed_everything(config.seed, workers=True)
    tokenizer = dataloader.get_tokenizer(config)
    logger = utils.get_logger(tag or 'experiment')

    # 加载模型
    model = diffusion.Diffusion(config, tokenizer=tokenizer).to('cuda')

    # 采样
    t0 = time.time()
    if config.sampling.visualize:
        samples, snapshots = model.restore_model_and_sample(
            num_steps=config.sampling.steps, ret_snapshots=True)
        model.display_snapshots(snapshots, max_length=80)
        model.save_snapshots_to_file(snapshots,
            f'snapshot_{tag}.txt', max_length=200)
        print(f'快照已保存到 snapshot_{tag}.txt')
    else:
        samples = model.restore_model_and_sample(
            num_steps=config.sampling.steps)
    elapsed = time.time() - t0

    text = model.tokenizer.batch_decode(samples)
    model.compute_generative_perplexity(text)
    ppl = model.gen_ppl_metric.compute().item()

    print(f'耗时: {elapsed:.1f}s | PPL: {ppl:.1f}')
    print(f'文本: {text[0][:200]}...')
    return elapsed, ppl, text, snapshots if config.sampling.visualize else None

---
## 实验 1a：多 Seed 可视化对比

同一设置下不同随机种子生成不同文本，展示扩散模型的非自回归随机性。

In [6]:
base_overrides = [
    'mode=sample_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-no_flashattn-fp32-owt',
    'data=openwebtext-split',
    'model.length=256',
    'sampling.predictor=ddpm_cache',
    'sampling.steps=1000',
    'loader.eval_batch_size=1',
    'sampling.num_sample_batches=1',
    'backbone=hf_dit',
    'sampling.visualize=True',
]

results_seeds = {}
for seed in [42, 123, 999]:
    print(f'\n{"="*60}')
    print(f'Seed = {seed}')
    print(f'{"="*60}')
    cfg = make_config(base_overrides + [f'seed={seed}'])
    elapsed, ppl, text, _ = run_experiment(cfg, tag=f'seed{seed}')
    results_seeds[seed] = {'time': elapsed, 'ppl': ppl, 'text': text[0]}

print('\n' + '=' * 60)
print('实验 1a 总结：多 Seed 对比')
print('=' * 60)
for seed, r in results_seeds.items():
    print(f'Seed {seed:3d} | 耗时 {r["time"]:.1f}s | PPL {r["ppl"]:.0f}')
    print(f'        文本: {r["text"][:120]}...')

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42



Seed = 42
配置路径: /content/mdlm_chinese/configs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possibl

扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.499 ( 50% 完成): <|endoftext|> 2015 [MASK] $ [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] 2016: $337...
t=0.399 ( 60% 完成): <|endoftext|> 2015 [MASK] $ [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] 2016: $337...
t=0.299 ( 70% 完成): <|endoftext|> 2015 [MASK] $ [MASK] [MASK] 938 [MASK] [MASK] 2016: $337, [MASK] [...
t=0.199 ( 80% 完成): <|endoftext|> 2015: $ [MASK] ,938 versus; 2016: $337, [MASK] [MASK] . [49 [MASK]...
t=0.149 ( 85% 完成): <|endoftext|> 2015: $ [MASK] ,938 versus; 2016: $337, [MASK] [MASK] . [49 [MASK]...
t=0.099 ( 90% 完成): <|endoftext|> 2015: $141,938 versus; 2016: $337, [MASK] [MASK] . [49 [MASK] On S...
t=0.079 ( 92% 完成): <|endoftext|> 2015: $141,938 versus; 2016: $337, [MASK] [MASK] . [49] On Sunday ...
t=0.059 ( 94% 完成): <|endoftext|> 2015: $141,938 versus; 2016: $337, [MASK] [MASK] . [49] On Sunday ...
t=0.039 ( 96% 完成): <|endoftext|> 2015: $141,938 ve

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

耗时: 18.4s | PPL: 52.5
文本: <|endoftext|> 2015: $141,938 versus; 2016: $337,746. [49]

On Sunday before the 12th game of preseason action, ESPN announced a new set of second-quarter projections for Buckman Field at Arizona's nex...

Seed = 123
配置路径: /content/mdlm_chinese/configs


INFO: Seed set to 123
INFO:lightning.fabric.utilities.seed:Seed set to 123


扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.499 ( 50% 完成): <|endoftext|> wants [MASK] [MASK] see [MASK] [MASK] So we identified [MASK] rang...
t=0.399 ( 60% 完成): <|endoftext|> wants students [MASK] see [MASK] . So we identified [MASK] range [...
t=0.299 ( 70% 完成): <|endoftext|> wants students [MASK] see [MASK] . So we identified [MASK] range [...
t=0.199 ( 80% 完成): <|endoftext|> wants students [MASK] see it. So we identified [MASK] range of pla...
t=0.149 ( 85% 完成): <|endoftext|> wants students to see it. So we identified [MASK] range of places ...
t=0.099 ( 90% 完成): <|endoftext|> wants students to see it. So we identified [MASK] range of places ...
t=0.079 ( 92% 完成): <|endoftext|> wants students to see it. So we identified [MASK] range of places ...
t=0.059 ( 94% 完成): <|endoftext|> wants students to see it. So we identified a range of places and r...
t=0.039 ( 96% 完成): <|endoftext|> wants students to

INFO: Seed set to 999
INFO:lightning.fabric.utilities.seed:Seed set to 999


扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.499 ( 50% 完成): 7.4 [MASK] .7 − [MASK] [MASK] 5 [MASK] [MASK] [MASK] [MASK] [MASK] 13 [MASK] 1) ...
t=0.399 ( 60% 完成): 7.4 [MASK] .7 − [MASK] [MASK] 5 [MASK] [MASK] [MASK] [MASK] 713 [MASK] 1) [MASK]...
t=0.299 ( 70% 完成): 7.4 40.7 − [MASK] [MASK] 5 [MASK] [MASK] [MASK] .713 [MASK] 1) [MASK] . [MASK] −...
t=0.199 ( 80% 完成): 7.4 40.7 −4 [MASK] 5 [MASK] ( [MASK] .713 [MASK] 1) [MASK] . [MASK] −0.0% [MASK]...
t=0.149 ( 85% 完成): 7.4 40.7 −4 [MASK] 5% ( [MASK] .713 [MASK] 1) [MASK] . [MASK] −0.0% [MASK] 0 [MA...
t=0.099 ( 90% 完成): 7.4 40.7 −4.5% ( [MASK] .713.1) [MASK] .9 −0.0% [MASK] 0 [MASK] 0 [MASK] 0.0 [MA...
t=0.079 ( 92% 完成): 7.4 40.7 −4.5% ( [MASK] .713.1) [MASK] .9 −0.0% [MASK] 0 [MASK] 0 [MASK] 0.0 [MA...
t=0.059 ( 94% 完成): 7.4 40.7 −4.5% ( [MASK] .713.1) 69.9 −0.0% [MASK] 0 [MASK] 0 [MASK] 0.0 [MASK] 0...
t=0.039 ( 96% 完成): 7.4 40.7 −4.5% ( [MASK] .713.1)

---
## 实验 1b：采样步数对比

固定 seed=42，对比 128 / 500 / 1000 步的生成质量差异。

In [7]:
results_steps = {}
for steps in [128, 500, 1000]:
    print(f'\n{"="*60}')
    print(f'Steps = {steps}')
    print(f'{"="*60}')
    cfg = make_config(base_overrides + [
        f'seed=42',
        f'sampling.steps={steps}',
    ])
    elapsed, ppl, text, _ = run_experiment(cfg, tag=f'steps{steps}')
    results_steps[steps] = {'time': elapsed, 'ppl': ppl, 'text': text[0]}

print('\n' + '=' * 60)
print('实验 1b 总结：步数对比 (seed=42)')
print('=' * 60)
print(f'{"步数":>6s} | {"耗时":>8s} | {"PPL":>8s} | 文本预览')
print(f'{"-"*6}-+-{"-"*8}-+-{"-"*8}-+-{"-"*30}')
for steps, r in sorted(results_steps.items()):
    print(f'{steps:6d} | {r["time"]:7.1f}s | {r["ppl"]:8.0f} | {r["text"][:80]}...')

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42



Steps = 128
配置路径: /content/mdlm_chinese/configs
扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.492 ( 51% 完成): [MASK] [MASK] [MASK] ball [MASK] out defeat [MASK] the Pet [MASK] Kus [MASK] [MA...
t=0.398 ( 60% 完成): [MASK] [MASK] [MASK] ball [MASK] out defeat [MASK] the Pet [MASK] Kusama [MASK] ...
t=0.297 ( 70% 完成): <|endoftext|> [MASK] -ball [MASK] out defeat [MASK] the Petro Kusama [MASK] , [M...
t=0.195 ( 80% 完成): <|endoftext|> dead-balling out defeat [MASK] the Petro Kusama [MASK] , [MASK] [M...
t=0.148 ( 85% 完成): <|endoftext|> dead-balling out defeat [MASK] the Petro Kusama Stadium, [MASK] [M...
t=0.094 ( 91% 完成): <|endoftext|> dead-balling out defeat [MASK] the Petro Kusama Stadium, before [M...
t=0.078 ( 92% 完成): <|endoftext|> dead-balling out defeat [MASK] the Petro Kusama Stadium, before [M...
t=0.055 ( 95% 完成): <|endoftext|> dead-balling out defeat [MASK] the Petro Kusama Stadium, before [M...
t

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


耗时: 3.5s | PPL: 136.9
文本: <|endoftext|> dead-balling out defeat at the Petro Kusama Stadium, before knocking himself off stage with his counter-punch that would pick new league hopes extinguished by the Kamenetians. Meanwhile,...

Steps = 500
配置路径: /content/mdlm_chinese/configs
扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.498 ( 50% 完成): [MASK] [MASK] [MASK] his [MASK] record to a [MASK] - [MASK] , $ [MASK] , [MASK] ...
t=0.398 ( 60% 完成): <|endoftext|> 2015 extending his [MASK] record to a three- [MASK] , $ [MASK] , [...
t=0.298 ( 70% 完成): <|endoftext|> 2015 extending his [MASK] record to a three- [MASK] , $337, [MASK]...
t=0.198 ( 80% 完成): <|endoftext|> 2015 extending his professional record to a three- [MASK] , $337, ...
t=0.148 ( 85% 完成): <|endoftext|> 2015 extending his professional record to a three- [MASK] , $337, ...
t=0.098 ( 90% 完成): <|endoftext|> 2015 extending his professional record to a thr

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


耗时: 7.2s | PPL: 79.1
文本: <|endoftext|> 2015 extending his professional record to a three-year, $337,501,000.[32] In his first ever season of 12,700 points (a projection under which no new contract-leveraged projections), Russ...

Steps = 1000
配置路径: /content/mdlm_chinese/configs
扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.499 ( 50% 完成): <|endoftext|> 2015 [MASK] $ [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] 2016: $337...
t=0.399 ( 60% 完成): <|endoftext|> 2015 [MASK] $ [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] 2016: $337...
t=0.299 ( 70% 完成): <|endoftext|> 2015 [MASK] $ [MASK] [MASK] 938 [MASK] [MASK] 2016: $337, [MASK] [...
t=0.199 ( 80% 完成): <|endoftext|> 2015: $ [MASK] ,938 versus; 2016: $337, [MASK] [MASK] . [49 [MASK]...
t=0.149 ( 85% 完成): <|endoftext|> 2015: $ [MASK] ,938 versus; 2016: $337, [MASK] [MASK] . [49 [MASK]...
t=0.099 ( 90% 完成): <|endoftext|> 2015: $141,938 versus; 2016: $337, [MASK] [MASK

---
## 实验 2：采样器效率对比

对比 ddpm_cache / ddpm / analytic 三种采样器的速度和 PPL。

**原理说明**:
- `ddpm_cache`: 缓存模型输出 p(x0|xt)，token 不变时跳过前向计算 → 最快
- `ddpm`: 标准 ancestral 采样，每步都跑网络 → 中等
- `analytic`: SEDD 式解析采样，使用得分函数 → 最慢

理论上同 seed 下三者应生成相同文本（等价性已验证）。

In [8]:
sampler_overrides = [
    'mode=sample_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-no_flashattn-fp32-owt',
    'data=openwebtext-split',
    'model.length=256',
    'sampling.steps=1000',
    'loader.eval_batch_size=1',
    'sampling.num_sample_batches=1',
    'backbone=hf_dit',
    'seed=42',
    'sampling.visualize=False',
]

results_samplers = {}
for predictor in ['ddpm_cache', 'ddpm', 'analytic']:
    print(f'\n{"="*60}')
    print(f'Sampler = {predictor}')
    print(f'{"="*60}')
    cfg = make_config(sampler_overrides + [
        f'sampling.predictor={predictor}',
    ])
    elapsed, ppl, text, _ = run_experiment(cfg, tag=predictor)
    results_samplers[predictor] = {'time': elapsed, 'ppl': ppl, 'text': text[0]}

print('\n' + '=' * 60)
print('实验 2 总结：采样器对比 (seed=42, steps=1000)')
print('=' * 60)
print(f'{"采样器":>12s} | {"耗时":>8s} | {"PPL":>8s} | 相对速度')
print(f'{"-"*12}-+-{"-"*8}-+-{"-"*8}-+-{"-"*10}')
fastest = min(r['time'] for r in results_samplers.values())
for name, r in results_samplers.items():
    speedup = r['time'] / fastest
    print(f'{name:>12s} | {r["time"]:7.1f}s | {r["ppl"]:8.0f} | {speedup:.2f}x')

# 验证三者生成文本是否一致
texts = [r['text'] for r in results_samplers.values()]
if len(set(texts)) == 1:
    print('\n✅ 三者生成文本完全相同（等价性验证通过）')
else:
    print('\n⚠️ 生成文本有差异（analytic 可能因浮点误差略有不同）')

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42



Sampler = ddpm_cache
配置路径: /content/mdlm_chinese/configs
耗时: 9.6s | PPL: 52.5
文本: <|endoftext|> 2015: $141,938 versus; 2016: $337,746. [49]

On Sunday before the 12th game of preseason action, ESPN announced a new set of second-quarter projections for Buckman Field at Arizona's nex...

Sampler = ddpm
配置路径: /content/mdlm_chinese/configs


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


耗时: 31.8s | PPL: 52.5
文本: <|endoftext|> 2015: $141,938 versus; 2016: $337,746. [49]

On Sunday before the 12th game of preseason action, ESPN announced a new set of second-quarter projections for Buckman Field at Arizona's nex...

Sampler = analytic
配置路径: /content/mdlm_chinese/configs
耗时: 36.5s | PPL: 52.5
文本: <|endoftext|> 2015: $141,938 versus; 2016: $337,746. [49]

On Sunday before the 12th game of preseason action, ESPN announced a new set of second-quarter projections for Buckman Field at Arizona's nex...

实验 2 总结：采样器对比 (seed=42, steps=1000)
         采样器 |       耗时 |      PPL | 相对速度
-------------+----------+----------+-----------
  ddpm_cache |     9.6s |       52 | 1.00x
        ddpm |    31.8s |       52 | 3.30x
    analytic |    36.5s |       52 | 3.79x

✅ 三者生成文本完全相同（等价性验证通过）


---
## 拓展实验：生成长文本 (Length=1024)

增加序列长度可以提升生成文本的连贯性。

In [11]:
cfg_long = make_config([
    'mode=sample_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-no_flashattn-fp32-owt',
    'data=openwebtext-split',
    'model.length=1024',
    'sampling.predictor=ddpm_cache',
    'sampling.steps=1000',
    'loader.eval_batch_size=1',
    'sampling.num_sample_batches=1',
    'backbone=hf_dit',
    'seed=42',
    'sampling.visualize=True',
])

print('生成长文本 (length=1024)，这可能需要 5-10 分钟...')
elapsed, ppl, text, _ = run_experiment(cfg_long, tag='length1024')
print(f'\n耗时: {elapsed:.1f}s | PPL: {ppl:.0f}')
print(f'\n完整文本:\n{text[0]}')

配置路径: /content/mdlm_chinese/configs


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


生成长文本 (length=1024)，这可能需要 5-10 分钟...
扩散逆向过程 (t=1.0 → t=0.0)
t=1.000 (  0% 完成): [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MA...
t=0.499 ( 50% 完成): [MASK] Trump [MASK] [MASK] � [MASK] [MASK] [MASK] � in the USA [MASK] [MASK] [MA...
t=0.399 ( 60% 完成): [MASK] Trump [MASK] ‘new [MASK] [MASK] � in the USA — TL [MASK] @freedom [MASK] ...
t=0.299 ( 70% 完成): [MASK] Trump [MASK] ‘new [MASK] ’ in the USA — TL [MASK] @freedom [MASK] ) Octob...
t=0.199 ( 80% 完成): [MASK] Trump [MASK] ‘new [MASK] ’ in the USA — TL [MASK] @freedom [MASK] ) Octob...
t=0.149 ( 85% 完成): <|endoftext|> Trump [MASK] ‘new [MASK] ’ in the USA — TL [MASK] @freedom [MASK] ...
t=0.099 ( 90% 完成): <|endoftext|> Trump fear ‘new normal’ in the USA — TL [MASK] @freedom [MASK] ) O...
t=0.079 ( 92% 完成): <|endoftext|> Trump fear ‘new normal’ in the USA — TL [MASK] @freedom [MASK] ) O...
t=0.059 ( 94% 完成): <|endoftext|> Trump fear ‘new normal’ in the USA — TLDC @freedomtoday) October 7...
t=0.039 ( 96%

---
## 结果汇总

所有快照文件已保存到当前目录：
- `snapshot_seed42.txt`, `snapshot_seed123.txt`, `snapshot_seed999.txt`
- `snapshot_steps128.txt`, `snapshot_steps500.txt`, `snapshot_steps1000.txt`
- `snapshot_length1024.txt`

下载这些文件即可用于课程报告插图。